# 🔃 Thử nghiệm Reranker cho kết quả truy xuất
**Mục tiêu:** Đánh giá xem sau khi có top-K kết quả từ Hybrid Search, việc dùng thêm một mô hình Reranker (sắp xếp lại) có cải thiện chất lượng không.

**Phương pháp:** Dùng `Gemini` làm LLM-as-a-Judge để cho điểm mức độ liên quan (relevance score 0-10) giữa câu hỏi và từng chunk, từ đó tái sắp xếp kết quả.

In [ ]:
import os, sys, json
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import google.generativeai as genai
from rank_bm25 import BM25Okapi
from source.core.config import Settings
from source.retrieval.hybrid_retriever import HybridRetriever

settings = Settings()
api_key = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=api_key)
llm = genai.GenerativeModel('gemini-2.0-flash')

retriever = HybridRetriever(settings=settings, collection_name="Traffic_Law_Hybrid")
CHUNKS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'chunks', 'traffic_chunks.json')
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    retriever.corpus_chunks = json.load(f)
tokenized_corpus = [retriever._tokenize(c['content']) for c in retriever.corpus_chunks]
retriever.bm25 = BM25Okapi(tokenized_corpus)

print(f"Đã tải {len(retriever.corpus_chunks):,} chunks | Gemini sẵn sàng!")

## 1. Định nghĩa LLM-as-a-Judge Reranker

In [ ]:
RERANK_PROMPT = """
Bạn là chuyên gia đánh giá mức độ liên quan của văn bản pháp lý.

Câu hỏi: "{query}"

Đoạn văn bản pháp lý:
\"\"\"{chunk}\"\"\"

Hãy chấm điểm mức độ liên quan từ 0 đến 10:
- 0: Hoàn toàn không liên quan
- 5: Có liên quan gián tiếp
- 10: Trực tiếp trả lời được câu hỏi

Chỉ trả về một con số nguyên duy nhất từ 0-10, không giải thích.
"""

def llm_rerank(query: str, results: list) -> list:
    """Rerank kết quả dùng Gemini làm judge."""
    reranked = []
    for r in results:
        try:
            prompt = RERANK_PROMPT.format(query=query, chunk=r['chunk']['content'][:500])
            response = llm.generate_content(prompt)
            score = float(response.text.strip())
        except:
            score = r['score'] * 10  # Fallback
        reranked.append({**r, 'rerank_score': score})
    return sorted(reranked, key=lambda x: x['rerank_score'], reverse=True)

print("LLM Reranker đã sẵn sàng!")

## 2. So sánh thứ tự trước và sau Rerank

In [ ]:
import numpy as np

def bm25_only_search(query: str, top_k: int = 8) -> list:
    """Search bằng BM25 thuần."""
    tokenized_q = retriever._tokenize(query)
    scores = retriever.bm25.get_scores(tokenized_q)
    if np.max(scores) > 0:
        scores = scores / np.max(scores)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [{"chunk": retriever.corpus_chunks[i], "score": float(scores[i])} for i in top_indices]

TEST_QUERY = "xe máy uống bia vượt đèn đỏ bị phạt thế nào?"
print(f" Câu hỏi thử nghiệm: '{TEST_QUERY}'\n")

# Lấy top-8 từ BM25
print(" Đang truy xuất top-8 từ BM25...")
initial_results = bm25_only_search(TEST_QUERY, top_k=8)

print("\n KẾT QUẢ TRƯỚC RERANK (thứ tự BM25):")
for i, r in enumerate(initial_results, 1):
    meta = r['chunk']['metadata']
    print(f"  [{i}] Score={r['score']:.3f} | {meta['dieu']} | {r['chunk']['content'][:90]}...")

In [ ]:
print(" Đang chạy LLM Reranker (Gemini)... (có thể mất 20-30 giây)")
reranked_results = llm_rerank(TEST_QUERY, initial_results)

print("\n KẾT QUẢ SAU RERANK (thứ tự Gemini):")
for i, r in enumerate(reranked_results, 1):
    meta = r['chunk']['metadata']
    original_rank = initial_results.index(next(x for x in initial_results if x['chunk']['content'] == r['chunk']['content'])) + 1
    rank_change = original_rank - i
    indicator = f"(↑{rank_change})" if rank_change > 0 else (f"(↓{abs(rank_change)})" if rank_change < 0 else "(=)")
    print(f"  [{i}] ReRank={r['rerank_score']:.0f}/10 {indicator} | {meta['dieu']} | {r['chunk']['content'][:90]}...")

## 3. Đo MRR (Mean Reciprocal Rank) trước và sau Rerank

In [ ]:
TEST_CASES_RERANK = [
    {"query": "nồng độ cồn xe máy bị phạt bao nhiêu tiền", "expected_dieu_keyword": "Điều 6"},
    {"query": "phạt xe không bảo hiểm", "expected_dieu_keyword": "bảo hiểm"},
    {"query": "mũ bảo hiểm không đúng quy chuẩn bị phạt", "expected_dieu_keyword": "mũ bảo hiểm"},
]

def mrr_score(results, keyword, k=5):
    for rank, r in enumerate(results[:k], 1):
        if keyword.lower() in r['chunk']['content'].lower() or keyword.lower() in r['chunk']['metadata']['dieu'].lower():
            return 1 / rank
    return 0.0

print(" So sánh MRR@5 Trước vs Sau Rerank:\n")
mrr_before_list = []
mrr_after_list = []

for tc in TEST_CASES_RERANK:
    before = bm25_only_search(tc['query'], top_k=5)
    after = llm_rerank(tc['query'], before)
    
    mrr_before = mrr_score(before, tc['expected_dieu_keyword'])
    mrr_after = mrr_score(after, tc['expected_dieu_keyword'])
    mrr_before_list.append(mrr_before)
    mrr_after_list.append(mrr_after)
    
    delta = mrr_after - mrr_before
    sign = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(f"  Q: {tc['query'][:50]}...")
    print(f"     MRR Trước: {mrr_before:.3f} → MRR Sau: {mrr_after:.3f}  {sign}{abs(delta):.3f}")
    print()

avg_before = sum(mrr_before_list) / len(mrr_before_list)
avg_after = sum(mrr_after_list) / len(mrr_after_list)
print(f"\n MRR@5 Trung bình:")
print(f"   BM25 Only  : {avg_before:.3f}")
print(f"   Sau Rerank : {avg_after:.3f}  {' Cải thiện!' if avg_after > avg_before else '👎 Không cải thiện'}")